In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

112

In [4]:
documents = documents_llm

In [5]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [7]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
import json
user_prompt = json.dumps(doc)

In [10]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [11]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [12]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [13]:
messages

[{'role': 'developer',
  'content': "You emulate a student who's taking our course.\nFormulate 5 questions this student might ask based on a FAQ record. The record\nshould contain the answer to the questions, and the questions should be complete and not too short.\nIf possible, use as fewer words as possible from the record.\n\nThe output should resemble how people ask questions\non the internet. Not too formal, not too short, not too long."},
 {'role': 'user',
  'content': '{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'}]

In [14]:
Questions

__main__.Questions

In [15]:
response

ParsedResponse[TypeVar](id='resp_0b84dffe50f7e939006a4fe864c674819bbc57145bc8122c62', created_at=1783621732.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ParsedResponseOutputMessage[TypeVar](id='msg_0b84dffe50f7e939006a4fe8654eb4819b86a1adb6f946e316', content=[ParsedResponseOutputText[TypeVar](annotations=[], text='{"questions":["I just found this course — is it too late to join, or can I still start now?","Can I enroll after the course has already started, or is registration closed?","If I join the course late, will I still be able to get a certificate?","What do I need to do to qualify for the certificate if I’m starting the course now?","Is it okay to begin the course after the deadline, and how does that affect the final project?"]}', type='output_text', logprobs=[], parsed=Questions(questions=['I just found this course — is it too late to join, or can I still start now?', 'Can I enroll after the 

In [16]:

response.output_parsed

Questions(questions=['I just found this course — is it too late to join, or can I still start now?', 'Can I enroll after the course has already started, or is registration closed?', 'If I join the course late, will I still be able to get a certificate?', 'What do I need to do to qualify for the certificate if I’m starting the course now?', 'Is it okay to begin the course after the deadline, and how does that affect the final project?'])

In [17]:

response.output_parsed.questions

['I just found this course — is it too late to join, or can I still start now?',
 'Can I enroll after the course has already started, or is registration closed?',
 'If I join the course late, will I still be able to get a certificate?',
 'What do I need to do to qualify for the certificate if I’m starting the course now?',
 'Is it okay to begin the course after the deadline, and how does that affect the final project?']

In [18]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [19]:
from evaluation_utils import llm_structured

In [20]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course—am I still allowed to join, or is it too late?', 'Can I enroll in the course after it has already started?', 'Is it okay to start the course late if I missed the beginning?', 'If I join the course now, can I still get a certificate?', 'What do I need to do to be eligible for the certificate if I’m joining late?']


In [21]:
result

Questions(questions=['I just found this course—am I still allowed to join, or is it too late?', 'Can I enroll in the course after it has already started?', 'Is it okay to start the course late if I missed the beginning?', 'If I join the course now, can I still get a certificate?', 'What do I need to do to be eligible for the certificate if I’m joining late?'])

In [22]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=93, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=300)

In [23]:
from evaluation_utils import calc_price

In [24]:
calc_price(usage)

{'input_cost': 0.00015525, 'output_cost': 0.0004185, 'total_cost': 0.00057375}

In [25]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course—am I still allowed to join, or is it too late?',
  'document': '74eb249bbf'},
 {'question': 'Can I enroll in the course after it has already started?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to start the course late if I missed the beginning?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course now, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for the certificate if I’m joining late?',
  'document': '74eb249bbf'}]

In [26]:
import pandas as pd

In [27]:

pd.DataFrame(records)

,question,document
0,I just found this course—am I still allowed to...,74eb249bbf
1,Can I enroll in the course after it has alread...,74eb249bbf
2,Is it okay to start the course late if I misse...,74eb249bbf
3,"If I join the course now, can I still get a ce...",74eb249bbf
4,What do I need to do to be eligible for the ce...,74eb249bbf


In [28]:
from evaluation_utils import llm_structured_retry

In [29]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [30]:
generate_ground_truth(doc)

([{'question': 'I found this course late — can I still join, or is it too late now?',
   'document': '74eb249bbf'},
  {'question': 'If I start the course after it already began, can I still get a certificate somehow?',
   'document': '74eb249bbf'},
  {'question': 'Do I need to submit the project before submissions close to qualify for the certificate?',
   'document': '74eb249bbf'},
  {'question': 'Is it okay to join the course anytime, or are there deadlines I should know about?',
   'document': '74eb249bbf'},
  {'question': 'What happens if I take the course now but miss the project submission window?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=101, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=308))

In [31]:
documents[:5]

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '489dd1c9d9',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer':

In [32]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [33]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [38]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/112 [00:00<?, ?it/s]

In [40]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

560

In [41]:
ground_truth[10]

{'question': 'How do students join the Office Hours or live workshop sessions if the Zoom link isn’t public?',
 'document': '489dd1c9d9'}

In [42]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08595449999999999

In [43]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08595449999999999

In [44]:
df_ground_truth = pd.DataFrame(ground_truth)

In [46]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [47]:
len(df_ground_truth)

560